In [2]:
from hattmapper import HATTMapper, PaulihedralDriver, TernaryTreeMapper, load_hamiltonian, load_molecule, pauli_weight, FermihedralMapper, HATTNaiveMapper
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import BravyiKitaevMapper, JordanWignerMapper
from prettytable import PrettyTable
from qiskit_nature.second_q.hamiltonians.lattices import (
    BoundaryCondition,
    SquareLattice,
)
from qiskit_nature.second_q.hamiltonians import FermiHubbardModel
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.transformers import FreezeCoreTransformer

## Table II

May take more than 1hr to compile and optimize circuits. The results are pretty printed in table format.

In [2]:
t = -1.0  # the interaction parameter
v = 0.0  # the onsite potential
u = 5.0  # the interaction parameter U

geometry = [
    (2, 2),
    (2, 3),
    (2, 4),
    (3, 3),
    (2, 5),
    (3, 4),
    (2, 7),
    (3, 5),
    (4, 4),
    (3, 6),
    (4, 5),
]

table = PrettyTable([f"{i}" for i in range(17)], header=False, hrules=True)
table.add_row(["", "", "Pauli Weight", "", "", "", "", "CNOT", "", "", "", "", "Depth", "", "", "", ""])
table.add_row(["Case", "Modes", "JW", "BK", "BTT", "FH", "HATT", "JW", "BK", "BTT", "FH", "HATT", "JW", "BK", "BTT", "FH", "HATT"])

for nrows, ncols in geometry:
    print(f"{nrows}x{ncols}")

    square_lattice = SquareLattice(
        rows=nrows, cols=ncols, boundary_condition=BoundaryCondition.PERIODIC
    )

    fhm = FermiHubbardModel(
        square_lattice.uniform_parameters(
            uniform_interaction=t,
            uniform_onsite_potential=v,
        ),
        onsite_interaction=u,
    )

    hamiltonian: FermionicOp = fhm.second_q_op().simplify()

    mappers = (
        JordanWignerMapper(),
        BravyiKitaevMapper(),
        TernaryTreeMapper(),
        FermihedralMapper.fermi_hubbard(),
        HATTMapper(hamiltonian)
    )

    weights = []
    cnots = []
    depths = []

    for m in mappers:
        weight = (
            "--"
            if isinstance(m, FermihedralMapper)
            and not m.solved(hamiltonian.register_length)
            else str(pauli_weight(hamiltonian, m))
        )

        weights.append(weight)

        complexity = (
            "--"
            if isinstance(m, FermihedralMapper)
            and not m.solved(hamiltonian.register_length)
            else PaulihedralDriver(hamiltonian, m).summary
        )

        if complexity == "--":
            cnots.append("--")
            depths.append("--")
        else:
            complexity = complexity.split("/")
            cnots.append(complexity[0])
            depths.append(complexity[2])

    table.add_row([f"{nrows}x{ncols}", hamiltonian.register_length, *weights, *cnots, *depths])

print(table)

2x2


2x3


2x4


3x3


2x5


...

3x5


4x4


3x6


4x5


+------+-------+--------------+------+-----+-----+------+------+-----+-----+-----+------+-------+-----+-----+-----+------+
|      |       | Pauli Weight |      |     |     |      | CNOT |     |     |     |      | Depth |     |     |     |      |
+------+-------+--------------+------+-----+-----+------+------+-----+-----+-----+------+-------+-----+-----+-----+------+
| Case | Modes |      JW      |  BK  | BTT |  FH | HATT |  JW  |  BK | BTT |  FH | HATT |   JW  |  BK | BTT |  FH | HATT |
+------+-------+--------------+------+-----+-----+------+------+-----+-----+-----+------+-------+-----+-----+-----+------+
| 2x2  |   8   |      80      |  80  |  86 |  56 |  76  |  51  |  71 |  77 |  37 |  61  |   60  |  97 | 106 |  35 |  69  |
+------+-------+--------------+------+-----+-----+------+------+-----+-----+-----+------+-------+-----+-----+-----+------+
| 2x3  |   12  |     212      | 200  | 199 | 161 | 187  | 159  | 172 | 161 | 123 | 139  |  160  | 219 | 228 | 155 | 191  |
+------+-------+

## Table III

May take more than 1hr to compile and optimize circuits. The results are pretty printed in table format.

**Note**: the order may be permuted

In [ ]:
table = PrettyTable([f"{i}" for i in range(14)], header=False, hrules=True)
table.add_row(["", "", "Pauli Weight", "", "", "", "CNOT", "", "", "", "Depth", "", "", ""])
table.add_row(["Case", "Modes", "JW", "BK", "BTT", "HATT", "JW", "BK", "BTT",  "HATT", "JW", "BK", "BTT", "HATT"])

NXS = (3, 4, 5, 6, 7)
NFS = (2, 3)

for nx in NXS:
    for nf in NFS:
        filename = f"tests/oneD_NX_{nx}_NF_{nf}.txt"
        
        print(f"NX = {nx}, NF = {nf}")

        hamiltonian: FermionicOp = load_hamiltonian(filename)

        mappers = (
            JordanWignerMapper(),
            BravyiKitaevMapper(),
            TernaryTreeMapper(),
            HATTMapper(hamiltonian)
        )

        weights = []
        cnots = []
        depths = []

        for m in mappers:
            weights.append(str(pauli_weight(hamiltonian, m)))
            complexity = PaulihedralDriver(hamiltonian, m).summary.split("/")
            cnots.append(complexity[0])
            depths.append(complexity[2])
        
        table.add_row([f"{nx}x{nf}F", hamiltonian.register_length, *weights, *cnots, *depths])

print(table)

## Table VI

In [4]:
table = PrettyTable([f"{i}" for i in range(3)], header=False, hrules=True)
table.add_row(["Case", "HATT (unopt)", "HATT"])

t = -1.0  # the interaction parameter
v = 0.0  # the onsite potential
u = 5.0  # the interaction parameter U


def testcase(casename: str, hamiltonian: FermionicOp):
    global table
    unopt_mapper = HATTNaiveMapper(hamiltonian)
    opt_mapper = HATTMapper(hamiltonian)
    table.add_row(
        [
            casename,
            str(pauli_weight(hamiltonian, unopt_mapper)),
            str(pauli_weight(hamiltonian, opt_mapper)),
        ]
    )


for atom, frzcore in [
    ("H_2", False),
    ("LiH", True),
    ("LiH", False),
    ("H_2O", False),
    ("CH_4", False),
    ("O_2", False),
]:
    problem = PySCFDriver(
        atom=load_molecule(f"tests/{atom}.json"),
        basis="sto3g",
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM,
    ).run()

    print(atom)

    if frzcore:
        problem = FreezeCoreTransformer(freeze_core=True).transform(problem)

    hamiltonian: FermionicOp = problem.hamiltonian.second_q_op()  # type: ignore

    testcase(f"{atom} sto3g" + (" frz" if frzcore else ""), hamiltonian)

for nrows, ncols in [(2, 2), (2, 3), (2, 4), (3, 3), (2, 5), (3, 4)]:
    square_lattice = SquareLattice(
        rows=nrows, cols=ncols, boundary_condition=BoundaryCondition.PERIODIC
    )

    print(f"{nrows}x{ncols}")

    fhm = FermiHubbardModel(
        square_lattice.uniform_parameters(
            uniform_interaction=t,
            uniform_onsite_potential=v,
        ),
        onsite_interaction=u,
    )

    hamiltonian: FermionicOp = fhm.second_q_op().simplify()
    testcase(f"{nrows}x{ncols}", hamiltonian)

for nx, nf in [(3, 2), (3, 3), (4, 2), (4, 3), (5, 2), (6, 2)]:
    filename = f"tests/oneD_NX_{nx}_NF_{nf}.txt"
    print(f"NX = {nx}, NF = {nf}")

    hamiltonian: FermionicOp = load_hamiltonian(filename)
    testcase(f"{nx}x{nf}F", hamiltonian)

print(table)

H_2


LiH


LiH


H_2O


CH_4


...

NX = 4, NF = 2


NX = 4, NF = 3


NX = 5, NF = 2


NX = 6, NF = 2


+---------------+--------------+-------+
|      Case     | HATT (unopt) |  HATT |
+---------------+--------------+-------+
|   H_2 sto3g   |      32      |   32  |
+---------------+--------------+-------+
| LiH sto3g frz |     1082     |  1082 |
+---------------+--------------+-------+
|   LiH sto3g   |     2926     |  2926 |
+---------------+--------------+-------+
|   H_2O sto3g  |    11944     | 11273 |
+---------------+--------------+-------+
|   CH_4 sto3g  |    37176     | 37083 |
+---------------+--------------+-------+
|   O_2 sto3g   |    13080     | 13084 |
+---------------+--------------+-------+
|      2x2      |      82      |   76  |
+---------------+--------------+-------+
|      2x3      |     194      |  187  |
+---------------+--------------+-------+
|      2x4      |     261      |  256  |
+---------------+--------------+-------+
|      3x3      |     404      |  410  |
+---------------+--------------+-------+
|      2x5      |     338      |  330  |
+---------------